# Surrogate Factory — UCCpHTP
## Chapter 2. Data Acquisition
Objectives:
- Load the pre-split HTP CFD dataset.
- Files expected in `UCCpHTP/data/`: `x_train.csv`, `x_val.csv`, `x_test.csv`, `yt_train.csv`, `yt_val.csv`, `yt_test.csv`.
- Merge x + yt to build Train / Val / Test sets.

### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
import pandas as pd
from pathlib import Path
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

### 2. Data Acquisition — Load pre-split files

In [ ]:
workflow.import_metadata(stage_name="SF_2_Data_Acquisition_Generation")

In [ ]:
import re

split_dir = Path(workflow.config['data_split_folder'])

# Names the rest of the pipeline uses (SF_5 / SF_6 metadata). The pre-split CSVs
# come straight from the CFD export, so their headers need not match: SF_5 was
# failing with KeyError: Index(['Cp']) because yt_*.csv carried a different name.
INPUTS  = ['x', 'y', 'z', 'alpha', 'mach']
OUTPUTS = ['Cp']


def load_split(filename, expected, what):
    """Read a split CSV, drop any written-out index column, and align headers."""
    df = pd.read_csv(split_dir / filename)

    # to_csv(index=True) upstream leaves an 'Unnamed: 0' column behind
    junk = [c for c in df.columns if re.fullmatch(r'Unnamed: \d+', str(c))]
    if junk:
        df = df.drop(columns=junk)
        print(f"  {filename}: dropped index column(s) {junk}")

    if list(df.columns) != expected:
        if len(df.columns) != len(expected):
            raise ValueError(
                f"{filename}: expected {len(expected)} {what} column(s) {expected}, "
                f"but the file has {len(df.columns)}: {list(df.columns)}.\n"
                f"Check the file, or edit INPUTS/OUTPUTS above to match your export."
            )
        print(f"  {filename}: renaming {list(df.columns)} -> {expected}")
        df = df.set_axis(expected, axis=1)

    return df


x_train = load_split('x_train.csv', INPUTS, 'input')
x_val   = load_split('x_val.csv',   INPUTS, 'input')
x_test  = load_split('x_test.csv',  INPUTS, 'input')

yt_train = load_split('yt_train.csv', OUTPUTS, 'output')
yt_val   = load_split('yt_val.csv',   OUTPUTS, 'output')
yt_test  = load_split('yt_test.csv',  OUTPUTS, 'output')

Train_set = pd.concat([x_train.reset_index(drop=True), yt_train.reset_index(drop=True)], axis=1)
Val_set   = pd.concat([x_val.reset_index(drop=True),   yt_val.reset_index(drop=True)],   axis=1)
Test_set  = pd.concat([x_test.reset_index(drop=True),  yt_test.reset_index(drop=True)],  axis=1)

for name, df in (('Train', Train_set), ('Val', Val_set), ('Test', Test_set)):
    missing = [c for c in INPUTS + OUTPUTS if c not in df.columns]
    if missing:
        raise ValueError(f"{name}_set is missing {missing} — got {list(df.columns)}")

print(f"\nTrain : {Train_set.shape[0]:>8,} rows   columns: {list(Train_set.columns)}")
print(f"Val   : {Val_set.shape[0]:>8,} rows")
print(f"Test  : {Test_set.shape[0]:>8,} rows")
Train_set.describe()


### Save

In [ ]:
job = workflow.config['job_name']
workflow.save_data(Train_set, job + '_Train_set.csv')
workflow.save_data(Val_set,   job + '_Val_set.csv')
workflow.save_data(Test_set,  job + '_Test_set.csv')
workflow.save_metadata()